In [58]:
import pandas as pd
import requests
import json
from datetime import datetime, timedelta
import smtplib
from email.message import EmailMessage
import os
from dotenv import load_dotenv
load_dotenv()

True

In [59]:
API_KEY = os.getenv("CURRENTS_API_KEY")
GMAIL_PASS = os.getenv("GMAIL_APP_PASS")
EMAIL_REMITENTE = os.getenv("EMAIL_REMITENTE")

In [60]:
usuarios = pd.read_csv('user_data_sample.csv', usecols=['user_id','nombre','apellido','email','pais'])
registro = pd.read_csv('user_register_data_sample.csv', usecols=['user_id','categoria_favorita','recibe_promociones'])
users = pd.merge(usuarios, registro, on='user_id', how='inner')
users_activos = users[users['recibe_promociones'] == True]
categorias = users_activos['categoria_favorita'].unique().tolist()

In [61]:

lista_noticias = []

PALABRAS_NEGATIVAS = [
    "muerte", "muerto", "fallece", "asesinato", "homicidio", 
    "crimen", "tiroteo", "tragedia", "accidente", "choque", 
    "guerra", "conflicto", "atentado", "bomba", "crisis",
    "ataque", "desastre", "catástrofe", "inundación", "huracán",
    "terremoto", "tsunami", "incendio", "explosión", "violencia", 
    "terrorismo", "pandemia", "epidemia", "desplazamiento", "refugiado", 
    "hambruna", "escasez", "contaminación", "derrame", "accidente nuclear",
    "desastre natural", "desplome", "colapso", "crisis", "destrucción", 
    "devastación", "cataclismo", "catástrofe ambiental", "desastre ecológico", 
    "desastre industrial", "atacaron", "asesinaron", "secuestraron", "violaron", "robado",
    "robaron", "saquearon", "destrozaron", "destruyeron", "devastaron"
]

for categoria in categorias:
    respuesta = requests.get(
        "https://api.currentsapi.services/v1/search",
        params={
            "category": categoria,
            "language": "es",
            "page_number": 1,
            "page_size": 10,
            "start_date": f"{(datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')}T12:00:00Z",
            "end_date": f"{datetime.now().strftime('%Y-%m-%d')}T23:59:59Z",
        },
        headers={"Authorization": API_KEY},
    )

    if respuesta.status_code == 200:
        datos = respuesta.json()
        noticias = datos.get("news", [])
        noticia_valida = next(
            (n for n in noticias if not any(p in n['title'].lower() for p in PALABRAS_NEGATIVAS)),
            noticias[0] if noticias else None
        )

        if noticia_valida:
            df_temp = pd.json_normalize([noticia_valida])
            df_temp['categoria_buscada'] = categoria
            lista_noticias.append(df_temp)
    else:
        print(f"Error en la solicitud: {respuesta.status_code}, categoría: {categoria}")

if lista_noticias:
    df_noticias = pd.concat(lista_noticias, ignore_index=True)
else:
    df_noticias = pd.DataFrame()

In [62]:
def enviar_noticia_usuario(user_id, df_users_activos, df_noticias):
    usuario = df_users_activos[df_users_activos['user_id'] == user_id]

    if usuario.empty:
        print(f"El usuario con ID {user_id} no existe o no tiene promociones activadas.")
        return False

    fila_user = usuario.iloc[0]
    nombre = fila_user['nombre']
    email_destinatario = fila_user['email']
    categoria_user = fila_user['categoria_favorita']

    noticias_cat = df_noticias[df_noticias['categoria_buscada'] == categoria_user]

    if noticias_cat.empty:
        print(f"No hay noticias en la categoría: {categoria_user}.")
        return False

    noticia = noticias_cat.iloc[0]
    titulo_noticia = noticia['title']
    link_noticia = noticia['url']

    msg = EmailMessage()
    msg["Subject"] = "Buenos Días, Buenas Noticias"
    msg["From"] = EMAIL_REMITENTE
    msg["To"] = email_destinatario

    cuerpo = (
        f"Buenos días {nombre},\n\n"
        f"Aquí tienes una noticia sobre {categoria_user}:\n\n"
        f"Título: {titulo_noticia}\n"
        f"Enlace a noticia: {link_noticia}\n\n"
    )
    msg.set_content(cuerpo)

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp:
        smtp.login(EMAIL_REMITENTE, GMAIL_PASS)
        smtp.send_message(msg)

    print(f"Correo enviado a {email_destinatario}.")
    return True

In [63]:
enviar_noticia_usuario(106, users_activos, df_noticias)

El usuario con ID 106 no existe o no tiene promociones activadas.


False